In [1]:
%pwd

'd:\\cmr\\Smart-Knowledge-Assistant\\knowledge'

In [2]:
import os
os.chdir("D:/cmr/Smart-Knowledge-Assistant")

In [3]:
%pwd

'D:\\cmr\\Smart-Knowledge-Assistant'

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\Pudhari Swaroopa\anaconda3\envs\smart-knowledge\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def load_pdf_files(data):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
        )
    documents=loader.load()
    return documents

In [6]:
extracted_data=load_pdf_files("data")

In [8]:
len(extracted_data)

339

In [9]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [10]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [11]:
def text_split(minimal_docs):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 800,
        chunk_overlap = 100
    )

    text_chunk = text_splitter.split_documents(minimal_docs)

    return text_chunk


In [12]:
text_chunk=text_split(minimal_docs)
print(f"Number of chunks: {len(text_chunk)}")

Number of chunks: 850


In [13]:
text_chunk = text_split(minimal_docs)

print(f"Number of chunks: {len(text_chunk)}")

Number of chunks: 850


In [14]:
text_chunk

[Document(metadata={'source': 'data\\CompleteAIML.pdf'}, page_content='ArtificiAl intelligence \nMAchine leArning \nAnd \ndeep leArning'),
 Document(metadata={'source': 'data\\CompleteAIML.pdf'}, page_content='LICENSE, DISCLAIMER OF LIABILITY, AND LIMITED WARRANTY\nBy purchasing or using this book and its companion files (the “Work”), you agree that \nthis license grants permission to use the contents contained herein, but does not give \nyou the right of ownership to any of the textual content in the book or ownership to \nany of the information, files, or products contained in it. This license does not permit \nuploading of the Work onto the Internet or on a network (of any kind) without the \nwritten consent of the Publisher. Duplication or dissemination of any text, code, simu -\nlations, images, etc. contained herein is limited to and subject to licensing terms for \nthe respective products, and permission must be obtained from the Publisher or the'),
 Document(metadata={'source':

In [15]:
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    #download and return the HuggingFace embeddings model.
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings
embedding = download_embeddings()

C:\Users\Pudhari Swaroopa\AppData\Local\Temp\ipykernel_19556\785579348.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [16]:
embedding


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [17]:
vector = embedding.embed_query("Hello World")
vector

[-0.03447723388671875,
 0.031023165211081505,
 0.006734973751008511,
 0.026108991354703903,
 -0.039362046867609024,
 -0.16030244529247284,
 0.06692394614219666,
 -0.0064415112137794495,
 -0.04745052382349968,
 0.014758817851543427,
 0.07087530195713043,
 0.05552754923701286,
 0.019193369895219803,
 -0.026251282542943954,
 -0.01010951679199934,
 -0.02694047801196575,
 0.022307446226477623,
 -0.022226616740226746,
 -0.1496925950050354,
 -0.017493078485131264,
 0.007676276378333569,
 0.054352231323719025,
 0.0032544422429054976,
 0.03172587975859642,
 -0.0846213772892952,
 -0.029405953362584114,
 0.05159562826156616,
 0.04812408611178398,
 -0.0033148485235869884,
 -0.05827920883893967,
 0.04196930304169655,
 0.022210706025362015,
 0.128188818693161,
 -0.02233896777033806,
 -0.011656241491436958,
 0.06292831897735596,
 -0.032876331359148026,
 -0.09122606366872787,
 -0.031175393611192703,
 0.05269952118396759,
 0.047034814953804016,
 -0.0842030942440033,
 -0.030056195333600044,
 -0.02074482

In [18]:
print(len(vector))

384


In [19]:
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY


In [20]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [21]:
pc


In [22]:
from pinecone import Pinecone, ServerlessSpec

index_name = "smart-knowledge-assistant"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [23]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name,
)

In [24]:
#load existing index
from langchain_community.vectorstores import FAISS
#embed each chunk and upsert the embeddings into your pinecone index.
docsearch =PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [25]:
retriever = docsearch.as_retriever(search_type="mmr", search_kwargs={"k":8})

In [26]:
retrieved_docs = retriever.invoke("What is AI")
retrieved_docs

[Document(metadata={'source': 'data\\CompleteAIML.pdf'}, page_content='Preface xv\nChapter 1:  Introduction to AI 1\n What is Artificial Intelligence?  2\nStrong AI versus Weak AI  4\n The Turing Test 5\nDefinition of the Turing Test 5\nAn Interrogator Test 6\n Heuristics 6\nGenetic Algorithms 8\n Knowledge Representation 8\nLogic-based Solutions 9\nSemantic Networks 9\n AI and Games 10\nThe Success of AlphaZero 11\n Expert Systems  12\n Neural Computing 13\n Evolutionary Computation 14\n Natural Language Processing 14\n Bioinformatics 17\n Major Parts of AI  18\nMachine Learning 18\nDeep Learning  19\nReinforcement Learning  19\nRobotics 20\n Code Samples 21\n Summary 22\nCONTENTS'),
 Document(metadata={'source': 'data\\CompleteAIML.pdf'}, page_content='the road. You might have a favorite ploy for recovering a dropped contact \nlens or for finding a parking space in a crowded shopping mall. Both are \nexamples of heuristics.\nAI problems tend to be large and computationally complex, a

In [27]:
from langchain_community.llms import Ollama
chatmodel = Ollama(model="llama3")

C:\Users\Pudhari Swaroopa\AppData\Local\Temp\ipykernel_19556\1239732716.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  chatmodel = Ollama(model="llama3")


In [28]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [29]:
system_prompt = (
    "You are an Smart Knowledge assistant for question answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [30]:
question_answer_chain = create_stuff_documents_chain(chatmodel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [32]:
user_question = input("Ask a question: ")

response = rag_chain.invoke({"input": user_question})

print("Assistant:", response["answer"])

Assistant: I can help with that! Based on the provided context, I can tell you that NLP (Natural Language Processing) involves interaction between computers and human languages, which is a field of computer science and artificial intelligence. It uses machine learning techniques to process and analyze natural language data, enabling tasks such as language translation, information retrieval, summarization, and sentiment analysis.
